# Brain Tumor DX - Universal GPU Training

Works on **Google Colab**, **Kaggle**, and **VS Code Colab extension**.

**Pipeline:**
1. Detect platform + GPU
2. Install dependencies
3. Setup classification data + project
4. Train classifier (ResNet50)
5. Evaluate classifier (accuracy, confusion matrix)
6. Prepare segmentation data from Kaggle (BraTS auto-detect)
7. Train segmenter (3D U-Net) with Dice/IoU monitoring
8. Evaluate segmenter (Dice, IoU, visualization)
9. Download checkpoints

In [ ]:
#@title Cell 1: Detect platform + verify GPU
import os, sys

PLATFORM = None
DATA_DIR = None
PROJECT_DIR = None
CHECKPOINT_DIR = None

if os.path.exists('/kaggle/working'):
    PLATFORM = 'kaggle'
    DATA_DIR = '/kaggle/input'
    PROJECT_DIR = '/kaggle/working/brain-tumor-dx'
    CHECKPOINT_DIR = '/kaggle/working/checkpoints'
elif os.path.exists('/content'):
    PLATFORM = 'colab'
    DATA_DIR = '/content/data'
    PROJECT_DIR = '/content/brain-tumor-dx'
    CHECKPOINT_DIR = '/content/checkpoints'
else:
    PLATFORM = 'local'
    DATA_DIR = './data'
    PROJECT_DIR = '.'
    CHECKPOINT_DIR = './checkpoints'

print(f'Platform: {PLATFORM}')
print(f'Data dir: {DATA_DIR}')
print(f'Project dir: {PROJECT_DIR}')
print(f'Checkpoint dir: {CHECKPOINT_DIR}')

import torch
print(f'\nPyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', None))
    if vram:
        print(f'VRAM: {vram / 1e9:.1f} GB')
    DEVICE = 'cuda'
else:
    print('WARNING: No GPU.')
    DEVICE = 'cpu'
os.environ['DEVICE'] = DEVICE
print(f'Device: {DEVICE}')

In [ ]:
#@title Cell 2: Install dependencies
!pip install -q torch torchvision monai nibabel pydicom scikit-image tqdm pydantic python-dotenv scikit-learn
print('Done.')

In [ ]:
#@title Cell 3: Setup classification data + project code
import os, subprocess, zipfile

CLASS_DATA = None

# Auto-find classification dataset
if PLATFORM == 'kaggle':
    for d in os.listdir(DATA_DIR):
        sub = os.path.join(DATA_DIR, d)
        if not os.path.isdir(sub):
            continue
        for cls in ['glioma', 'meningioma', 'pituitary', 'notumor']:
            if os.path.exists(os.path.join(sub, cls)):
                CLASS_DATA = sub
                print(f'Classification data: {sub}')
                break
        if CLASS_DATA is None:
            if os.path.isdir(os.path.join(sub, 'Training')):
                CLASS_DATA = sub
                print(f'Classification data: {sub}')
                break

elif PLATFORM == 'colab':
    zips = [f for f in os.listdir('/content')
            if f.endswith('.zip') and os.path.isfile(f'/content/{f}')]
    if not zips:
        try:
            from google.colab import files
            print('Upload classification .zip:')
            uploaded = files.upload()
            for fname in uploaded.keys():
                zips.append(fname)
        except ImportError:
            print('No zips. Upload via Files panel then re-run.')
    for zpath in zips:
        print(f'Extracting {zpath}...')
        with zipfile.ZipFile(f'/content/{zpath}', 'r') as z:
            z.extractall(DATA_DIR)
        os.remove(f'/content/{zpath}')
    CLASS_DATA = DATA_DIR

if CLASS_DATA:
    for folder_name in ['Training', 'Testing']:
        src = os.path.join(CLASS_DATA, folder_name)
        if os.path.isdir(src):
            for item in os.listdir(src):
                s = os.path.join(src, item)
                d = os.path.join(CLASS_DATA, item)
                if not os.path.exists(d):
                    os.rename(s, d)
            os.rmdir(src)
    notumor = os.path.join(CLASS_DATA, 'notumor')
    no_tumor = os.path.join(CLASS_DATA, 'no_tumor')
    if os.path.isdir(notumor) and not os.path.isdir(no_tumor):
        os.rename(notumor, no_tumor)

if CLASS_DATA:
    for cls in ['glioma', 'meningioma', 'pituitary', 'no_tumor']:
        cls_path = os.path.join(CLASS_DATA, cls)
        if os.path.isdir(cls_path):
            n = len([x for x in os.listdir(cls_path) if os.path.isfile(os.path.join(cls_path, x))])
            print(f'  {cls}/: {n} images')

# Project code
if not os.path.exists(f'{PROJECT_DIR}/src/brain_tumor_dx'):
    print(f'Project not found at {PROJECT_DIR}')
    print('Upload brain-tumor-dx.zip or push to GitHub.')
else:
    print(f'Project: {PROJECT_DIR}')

src_path = f'{PROJECT_DIR}/src'
if src_path not in sys.path:
    sys.path.insert(0, src_path)

if os.path.exists(f'{PROJECT_DIR}/src/brain_tumor_dx'):
    from brain_tumor_dx.config import settings
    settings.device = DEVICE
    print(f'Device: {settings.device}')
    print(f'Classes: {settings.tumor_classes}')
    print('Setup OK.')
else:
    print('WARNING: Project code not found.')

In [ ]:
#@title Cell 4: Train CLASSIFIER (ResNet50)
import os, torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from brain_tumor_dx.config import settings
from brain_tumor_dx.data.datasets import ClassificationDataset
from brain_tumor_dx.models.classifier import TumorClassifier

DATA_ROOT = CLASS_DATA  #@param {type:"string"}
EPOCHS = 15  #@param {type:"integer"}
BATCH_SIZE = 32  #@param {type:"integer"}
LR = 1e-4  #@param {type:"number"}

for cls in settings.tumor_classes:
    cls_dir = os.path.join(DATA_ROOT, cls)
    if not os.path.isdir(cls_dir):
        raise FileNotFoundError(f'Missing: {cls_dir}')
    n = len([f for f in os.listdir(cls_dir) if os.path.isfile(os.path.join(cls_dir, f))])
    print(f'  {cls}: {n} images')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
OUT = os.path.join(CHECKPOINT_DIR, 'classifier.pt')

dataset = ClassificationDataset(DATA_ROOT)
print(f'Total: {len(dataset)} samples')
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

model = TumorClassifier(num_classes=len(settings.tumor_classes)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = torch.nn.CrossEntropyLoss()

train_losses = []
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(loader, desc=f'epoch {epoch+1}/{EPOCHS}'):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg = running_loss / len(loader)
    train_losses.append(avg)
    print(f'epoch {epoch+1}: loss={avg:.4f}')

torch.save(model.state_dict(), OUT)
print(f'Saved -> {OUT}')

In [ ]:
#@title Cell 5: Evaluate classifier
import os, torch, numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

from brain_tumor_dx.config import settings
from brain_tumor_dx.data.datasets import ClassificationDataset
from brain_tumor_dx.models.classifier import TumorClassifier

CKPT = os.path.join(CHECKPOINT_DIR, 'classifier.pt')
model = TumorClassifier(num_classes=len(settings.tumor_classes)).to(DEVICE)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()

dataset = ClassificationDataset(DATA_ROOT)
loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(loader, desc='Evaluating'):
        images = images.to(DEVICE)
        outputs = model(images)
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds, all_labels = np.array(all_preds), np.array(all_labels)
classes = list(settings.tumor_classes)

print('=' * 60)
print('         CLASSIFICATION EVALUATION')
print('=' * 60)
print(f'  Accuracy:  {accuracy_score(all_labels, all_preds):.4f} ({accuracy_score(all_labels, all_preds)*100:.2f}%)')
print(f'  Precision: {precision_score(all_labels, all_preds, average="macro"):.4f} (macro)')
print(f'  Recall:    {recall_score(all_labels, all_preds, average="macro"):.4f} (macro)')
print(f'  F1 Score:  {f1_score(all_labels, all_preds, average="macro"):.4f} (macro)')
print('\n' + classification_report(all_labels, all_preds, target_names=classes))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes).plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

fig2, ax2 = plt.subplots(figsize=(8, 4))
ax2.plot(range(1, len(train_losses) + 1), train_losses, 'b-o', markersize=4)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Training Loss')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#@title Cell 6: Prepare segmentation data (BraTS from Kaggle)
import os, glob, numpy as np

SEG_DATA = None

# --- Auto-find segmentation data on Kaggle ---
if PLATFORM == 'kaggle':
    print('Searching /kaggle/input for segmentation data...')
    for d in os.listdir(DATA_DIR):
        sub = os.path.join(DATA_DIR, d)
        if not os.path.isdir(sub):
            continue
        print(f'  Checking: {d}')
        # Format A: case_XXX/image.nii.gz (converted NIfTI)
        if os.path.exists(os.path.join(sub, 'case_001', 'image.nii.gz')):
            SEG_DATA = sub
            print(f'  -> Found converted NIfTI format')
            break
        # Format B: BraTS2020 raw .nii.gz files (BraTS20_Training_001_*.nii.gz)
        raw_files = glob.glob(os.path.join(sub, 'BraTS20*_flair.nii.gz'))
        if raw_files:
            SEG_DATA = sub
            print(f'  -> Found raw BraTS format')
            break
        # Format C: MONAI Task format (imagesTr/labelsTr)
        if os.path.isdir(os.path.join(sub, 'imagesTr')):
            SEG_DATA = sub
            print(f'  -> Found MONAI Task format (imagesTr/)')
            break
        # Check inside subfolders
        for sub2 in os.listdir(sub):
            sub2_path = os.path.join(sub, sub2)
            if os.path.isdir(sub2_path):
                if os.path.exists(os.path.join(sub2_path, 'case_001', 'image.nii.gz')):
                    SEG_DATA = sub2_path
                    print(f'  -> Found converted NIfTI format in {sub2}')
                    break
                raw_files = glob.glob(os.path.join(sub2_path, 'BraTS20*_flair.nii.gz'))
                if raw_files:
                    SEG_DATA = sub2_path
                    print(f'  -> Found raw BraTS format in {sub2}')
                    break
                if os.path.isdir(os.path.join(sub2_path, 'imagesTr')):
                    SEG_DATA = sub2_path
                    print(f'  -> Found MONAI Task format in {sub2}')
                    break

elif PLATFORM == 'colab':
    print('Looking for segmentation data in /content/data/...')
    if os.path.isdir(DATA_DIR):
        for d in os.listdir(DATA_DIR):
            sub = os.path.join(DATA_DIR, d)
            if not os.path.isdir(sub):
                continue
            if os.path.exists(os.path.join(sub, 'case_001', 'image.nii.gz')):
                SEG_DATA = sub
                break
            raw_files = glob.glob(os.path.join(sub, 'BraTS20*_flair.nii.gz'))
            if raw_files:
                SEG_DATA = sub
                break

# --- Helper: find all NIfTI slices from a directory of raw BraTS volumes ---
def prepare_raw_brats(raw_dir, output_dir):
    """Convert raw BraTS .nii.gz files to case_XXX/{image,mask}.nii.gz"""
    import nibabel as nib
    os.makedirs(output_dir, exist_ok=True)
    raw_files = sorted(glob.glob(os.path.join(raw_dir, 'BraTS20*_flair.nii.gz')))
    for i, flair_path in enumerate(raw_files):
        prefix = flair_path.replace('_flair.nii.gz', '')
        case_name = f'case_{i+1:03d}'
        case_dir = os.path.join(output_dir, case_name)
        os.makedirs(case_dir, exist_ok=True)
        out_img = os.path.join(case_dir, 'image.nii.gz')
        out_msk = os.path.join(case_dir, 'mask.nii.gz')
        if os.path.exists(out_img) and os.path.exists(out_msk):
            continue
        # Stack modalities: FLAIR + T1 + T1ce + T2 -> 4-channel image
        flair = nib.load(flair_path).get_fdata().astype(np.float32)
        t1_path = flair_path.replace('_flair', '_t1')
        t1ce_path = flair_path.replace('_flair', '_t1ce')
        t2_path = flair_path.replace('_flair', '_t2')
        seg_path = flair_path.replace('_flair', '_seg')
        modalities = [flair]
        for mp in [t1_path, t1ce_path, t2_path]:
            if os.path.exists(mp):
                modalities.append(nib.load(mp).get_fdata().astype(np.float32))
            else:
                modalities.append(np.zeros_like(flair))
        image_data = np.stack(modalities, axis=0)  # (4, H, W, D)
        nib.save(nib.Nifti1Image(image_data, np.eye(4)), out_img)
        # Mask: binary tumor
        if os.path.exists(seg_path):
            mask_data = nib.load(seg_path).get_fdata()
            mask_binary = (mask_data > 0).astype(np.float32)
        else:
            mask_binary = np.zeros(flair.shape, dtype=np.float32)
        nib.save(nib.Nifti1Image(mask_binary[np.newaxis], np.eye(4)), out_msk)
        print(f'  {case_name}: {image_data.shape}')
    return output_dir

# --- Convert if raw format detected ---
CONVERTED_DIR = os.path.join(DATA_DIR, 'segmentation_converted')

if SEG_DATA:
    raw_files = glob.glob(os.path.join(SEG_DATA, 'BraTS20*_flair.nii.gz'))
    if raw_files:
        print(f'\nConverting {len(raw_files)} raw BraTS volumes to NIfTI...')
        SEG_DATA = prepare_raw_brats(SEG_DATA, CONVERTED_DIR)
    elif os.path.isdir(os.path.join(SEG_DATA, 'imagesTr')):
        print(f'\nConverting MONAI Task format to NIfTI...')
        import nibabel as nib
        os.makedirs(CONVERTED_DIR, exist_ok=True)
        img_dir = os.path.join(SEG_DATA, 'imagesTr')
        lbl_dir = os.path.join(SEG_DATA, 'labelsTr') if os.path.isdir(os.path.join(SEG_DATA, 'labelsTr')) else None
        img_files = sorted(glob.glob(os.path.join(img_dir, '*.nii.gz')))
        for i, img_path in enumerate(img_files):
            case_name = f'case_{i+1:03d}'
            case_dir = os.path.join(CONVERTED_DIR, case_name)
            os.makedirs(case_dir, exist_ok=True)
            nib.save(nib.load(img_path), os.path.join(case_dir, 'image.nii.gz'))
            if lbl_dir:
                lbl_path = os.path.join(lbl_dir, os.path.basename(img_path))
                if os.path.exists(lbl_path):
                    nib.save(nib.load(lbl_path), os.path.join(case_dir, 'mask.nii.gz'))
        SEG_DATA = CONVERTED_DIR

# --- Verify and report ---
if SEG_DATA and os.path.exists(os.path.join(SEG_DATA, 'case_001', 'image.nii.gz')):
    n_cases = len([d for d in os.listdir(SEG_DATA) if os.path.isdir(os.path.join(SEG_DATA, d))])
    print(f'\nSegmentation data ready: {n_cases} cases at {SEG_DATA}')
    import nibabel as nib
    sample_img = nib.load(os.path.join(SEG_DATA, 'case_001', 'image.nii.gz'))
    sample_msk = nib.load(os.path.join(SEG_DATA, 'case_001', 'mask.nii.gz'))
    print(f'  Volume shape: {sample_img.shape}')
    print(f'  Mask shape:   {sample_msk.shape}')
    mask_vals = sorted(set(sample_msk.get_fdata().flatten().astype(int)))
    print(f'  Mask values:  {mask_vals}')
else:
    print('\nNo segmentation data found.')
    print('Add a BraTS dataset to this notebook (Settings > Data > Add data)')
    print('Supported formats:')
    print('  1. Converted: case_001/{image.nii.gz, mask.nii.gz}')
    print('  2. Raw BraTS: BraTS20_Training_001_{flair,t1,t1ce,t2,seg}.nii.gz')
    print('  3. MONAI Task: imagesTr/*.nii.gz + labelsTr/*.nii.gz')

In [ ]:
#@title Cell 7: Train SEGMENTER (3D U-Net) with Dice + IoU monitoring
import os, torch, numpy as np
from torch.utils.data import DataLoader
from monai.losses import DiceLoss
from tqdm import tqdm

from brain_tumor_dx.config import settings
from brain_tumor_dx.data.datasets import SegmentationDataset
from brain_tumor_dx.models.segmentation import TumorSegmenter

if not SEG_DATA or not os.path.isdir(SEG_DATA):
    raise FileNotFoundError('No segmentation data. Run Cell 6 first.')

EPOCHS = 50  #@param {type:"integer"}
BATCH_SIZE = 2  #@param {type:"integer"}
LR = 1e-4  #@param {type:"number"}
VAL_SPLIT = 0.2  #@param {type:"number"}

dataset = SegmentationDataset(SEG_DATA)
n_val = int(len(dataset) * VAL_SPLIT)
n_train = len(dataset) - n_val
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [n_train, n_val])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)

print(f'Train: {n_train} cases, Val: {n_val} cases')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
OUT = os.path.join(CHECKPOINT_DIR, 'segmentation.pt')

model = TumorSegmenter().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = DiceLoss(sigmoid=True)

def compute_dice(pred, target, threshold=0.5):
    pred_bin = (torch.sigmoid(pred) > threshold).float()
    intersection = (pred_bin * target).sum()
    union = pred_bin.sum() + target.sum()
    return float((2 * intersection + 1e-8) / (union + 1e-8))

def compute_iou(pred, target, threshold=0.5):
    pred_bin = (torch.sigmoid(pred) > threshold).float()
    intersection = (pred_bin * target).sum()
    union = pred_bin.sum() + target.sum() - intersection
    return float((intersection + 1e-8) / (union + 1e-8))

train_losses = []
val_dices = []
val_ious = []
best_dice = 0.0

for epoch in range(EPOCHS):
    # Training
    model.train()
    running_loss = 0.0
    for images, masks in tqdm(train_loader, desc=f'epoch {epoch+1}/{EPOCHS} [train]'):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        pred = model(images)
        loss = criterion(pred, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)

    # Validation
    model.eval()
    val_dice_sum, val_iou_sum = 0.0, 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            pred = model(images)
            val_dice_sum += compute_dice(pred, masks)
            val_iou_sum += compute_iou(pred, masks)
    avg_dice = val_dice_sum / len(val_loader)
    avg_iou = val_iou_sum / len(val_loader)
    val_dices.append(avg_dice)
    val_ious.append(avg_iou)

    marker = ''
    if avg_dice > best_dice:
        best_dice = avg_dice
        torch.save(model.state_dict(), OUT)
        marker = ' [BEST]'

    print(f'epoch {epoch+1}: loss={avg_loss:.4f} | val_dice={avg_dice:.4f} | val_iou={avg_iou:.4f}{marker}')

print(f'\nTraining complete. Best val Dice: {best_dice:.4f}')
print(f'Saved -> {OUT}')

In [ ]:
#@title Cell 8: Evaluate segmenter (Dice, IoU, visualization)
import os, torch, numpy as np
import matplotlib.pyplot as plt

from brain_tumor_dx.config import settings
from brain_tumor_dx.data.datasets import SegmentationDataset
from brain_tumor_dx.models.segmentation import TumorSegmenter

CKPT = os.path.join(CHECKPOINT_DIR, 'segmentation.pt')
model = TumorSegmenter().to(DEVICE)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()

dataset = SegmentationDataset(SEG_DATA)
loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=2)

all_dices, all_ious = [], []
per_case_results = []

with torch.no_grad():
    for i, (images, masks) in enumerate(tqdm(loader, desc='Evaluating')):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        pred = model(images)
        pred_bin = (torch.sigmoid(pred) > 0.5).float()

        intersection = (pred_bin * masks).sum().item()
        dice = (2 * intersection + 1e-8) / (pred_bin.sum().item() + masks.sum().item() + 1e-8)
        iou_val = (intersection + 1e-8) / (pred_bin.sum().item() + masks.sum().item() - intersection + 1e-8)

        all_dices.append(dice)
        all_ious.append(iou_val)
        per_case_results.append({'case': i+1, 'dice': dice, 'iou': iou_val})

# Summary
print('=' * 60)
print('         SEGMENTATION EVALUATION')
print('=' * 60)
print(f'  Total cases:     {len(all_dices)}')
print(f'  Mean Dice:       {np.mean(all_dices):.4f} (+/- {np.std(all_dices):.4f})')
print(f'  Mean IoU:        {np.mean(all_ious):.4f} (+/- {np.std(all_ious):.4f})')
print(f'  Median Dice:     {np.median(all_dices):.4f}')
print(f'  Median IoU:      {np.median(all_ious):.4f}')
print(f'  Min Dice:        {np.min(all_dices):.4f}')
print(f'  Max Dice:        {np.max(all_dices):.4f}')
print(f'  Dice >= 0.9:     {sum(1 for d in all_dices if d >= 0.9)}/{len(all_dices)}')
print(f'  Dice >= 0.8:     {sum(1 for d in all_dices if d >= 0.8)}/{len(all_dices)}')
print(f'  Dice >= 0.5:     {sum(1 for d in all_dices if d >= 0.5)}/{len(all_dices)}')

# Plot metrics distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(all_dices, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(all_dices), color='red', linestyle='--', label=f'Mean: {np.mean(all_dices):.3f}')
axes[0].set_title('Dice Score Distribution')
axes[0].set_xlabel('Dice')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(all_ious, bins=20, color='coral', edgecolor='black', alpha=0.7)
axes[1].axvline(np.mean(all_ious), color='red', linestyle='--', label=f'Mean: {np.mean(all_ious):.3f}')
axes[1].set_title('IoU Distribution')
axes[1].set_xlabel('IoU')
axes[1].legend()

epochs_range = range(1, len(val_dices) + 1)
axes[2].plot(epochs_range, train_losses, 'b-', label='Train Loss')
axes[2].plot(epochs_range, val_dices, 'g-', label='Val Dice')
axes[2].plot(epochs_range, val_ious, 'r-', label='Val IoU')
axes[2].set_title('Training Progress')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Visualize predictions on sample cases
fig2, axes2 = plt.subplots(3, 6, figsize=(18, 9))
sample_indices = np.linspace(0, len(dataset)-1, 3, dtype=int)

for row, idx in enumerate(sample_indices):
    image, mask = dataset[idx]
    image_tensor = image.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = torch.sigmoid(model(image_tensor)).squeeze().cpu().numpy()
    pred_bin = (pred > 0.5).astype(np.float32)

    img_2d = image.squeeze().numpy()
    mid = img_2d.shape[2] // 2

    axes2[row, 0].imshow(img_2d[0, :, :, mid], cmap='gray')
    axes2[row, 0].set_title('Image')
    axes2[row, 1].imshow(mask.squeeze().numpy()[:, :, mid], cmap='gray')
    axes2[row, 1].set_title('Ground Truth')
    axes2[row, 2].imshow(pred_bin[:, :, mid], cmap='gray')
    axes2[row, 2].set_title('Prediction')

    # Overlay
    axes2[row, 3].imshow(img_2d[0, :, :, mid], cmap='gray')
    axes2[row, 3].imshow(mask.squeeze().numpy()[:, :, mid], cmap='Reds', alpha=0.4)
    axes2[row, 3].set_title('GT Overlay')

    axes2[row, 4].imshow(img_2d[0, :, :, mid], cmap='gray')
    axes2[row, 4].imshow(pred_bin[:, :, mid], cmap='Blues', alpha=0.4)
    axes2[row, 4].set_title('Pred Overlay')

    # Difference
    diff = mask.squeeze().numpy()[:, :, mid] - pred_bin[:, :, mid]
    axes2[row, 5].imshow(diff, cmap='RdBu_r', vmin=-1, vmax=1)
    axes2[row, 5].set_title('Difference')

for ax in axes2.flat:
    ax.axis('off')
plt.suptitle('Segmentation Predictions vs Ground Truth')
plt.tight_layout()
plt.show()

In [ ]:
#@title Cell 9: Download all checkpoints
import os

for name in ['classifier.pt', 'segmentation.pt']:
    CKPT = os.path.join(CHECKPOINT_DIR, name)
    if not os.path.exists(CKPT):
        print(f'{name}: not found')
        continue
    print(f'{name}: {CKPT}')
    if PLATFORM == 'colab':
        from google.colab import files
        files.download(CKPT)
    elif PLATFORM == 'kaggle':
        print('  Download from Output tab.')
    else:
        print('  Copy from checkpoints/')